In [0]:
# GDPR and PII Handling Review

from pyspark.sql import Row

CATALOG = "insurance_lakehouse"

silver_customers = spark.table(f"{CATALOG}.silver.silver_customers")
silver_payments = spark.table(f"{CATALOG}.silver.silver_payments")

customer_columns = silver_customers.columns
payment_columns = silver_payments.columns

raw_pii_customer_fields = [
    "first_name",
    "last_name",
    "email",
    "phone_number",
    "street",
    "postal_code",
    "date_of_birth"
]

hashed_customer_fields = [
    "customer_hash",
    "email_hash",
    "phone_hash"
]

pii_check_rows = []

for field in raw_pii_customer_fields:
    pii_check_rows.append(
        Row(
            dataset="silver_customers",
            field=field,
            expected_handling="not exposed in Silver",
            present_in_silver=field in customer_columns,
            status="PASS" if field not in customer_columns else "CHECK"
        )
    )

for field in hashed_customer_fields:
    pii_check_rows.append(
        Row(
            dataset="silver_customers",
            field=field,
            expected_handling="hashed/pseudonymized field exists",
            present_in_silver=field in customer_columns,
            status="PASS" if field in customer_columns else "CHECK"
        )
    )

pii_check_rows.append(
    Row(
        dataset="silver_payments",
        field="iban_hash",
        expected_handling="hashed bank-like identifier only",
        present_in_silver="iban_hash" in payment_columns,
        status="PASS" if "iban_hash" in payment_columns else "CHECK"
    )
)

pii_check_df = spark.createDataFrame(pii_check_rows)

display(pii_check_df)